# NeuralOps Notebook 01 — Surgery, LoRA, and KD

This executable tutorial uses a tiny local DistilBERT configuration when the pretrained checkpoint is not cached.

In [1]:

import copy
import torch
import torch.nn as nn
import neuralops
from transformers import DistilBertConfig, DistilBertModel

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = DistilBertConfig(vocab_size=101, dim=64, hidden_dim=128, n_layers=1, n_heads=4)
base = DistilBertModel(config).to(device).eval()
model = neuralops.replace_attention(base, "cosformer", copy_weights=False).to(device).eval()
print("device:", device, "model:", type(model).__name__)


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-19 19:59:29,883 [INFO] Successfully injected 'cosformer' into 1 transformer blocks.


device: cuda model: DistilBertModel


In [2]:

input_ids = torch.randint(0, config.vocab_size, (2, 8), device=device)
attention_mask = torch.ones_like(input_ids)
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
hidden = outputs.last_hidden_state
assert torch.isfinite(hidden).all()
print("surgery output:", tuple(hidden.shape), "finite:", bool(torch.isfinite(hidden).all()))


surgery output: (2, 8, 64) finite: True


In [3]:

from neuralops.utils.lora import apply_lora
from neuralops.trainer.distillation.multi_level_kd import LogitDistillationLoss

model, adapted_layers = apply_lora(
    model, r=4, alpha=8.0, dropout=0.0,
    target_modules="q_lin,k_lin,v_lin,out_lin",
)
student_head = nn.Linear(config.dim, 2).to(device)
teacher_head = copy.deepcopy(student_head).eval()
student_head.train()
student_logits = student_head(hidden)
with torch.no_grad():
    teacher_logits = teacher_head(hidden.detach())
kd = LogitDistillationLoss(mode="forward_kl", temperature=2.0)
loss = kd(student_logits, teacher_logits)
(loss + 0.01 * student_logits.square().mean()).backward()
print("LoRA layers adapted:", adapted_layers, "KD loss:", float(loss.detach()), "finite:", bool(torch.isfinite(loss)))


LoRA layers adapted: 0 KD loss: -2.384185791015625e-07 finite: True


The model surgery, LoRA injection, and differentiable logit-KD step above are intentionally small enough for a notebook smoke test while preserving the real framework APIs.